In [ ]:
import gzip
import json

meta_file = '/data/mhwang/LLM/Fre_LLM/dataset/raw_data/Movies.json.gz'
title_map_file = '/data/mhwang/LLM/Fre_LLM/dataset/process_data/Movies/item_titles.json'

title_map = {}

with gzip.open(meta_file, 'rt', encoding='utf-8') as f:
    for line in f:
        data = json.loads(line.strip())
        asin = data.get('asin')
        title = data.get('title', 'Unknown')
        if asin:
            title_map[asin] = title

# 保存为 JSON 文件备用
with open(title_map_file, 'w', encoding='utf-8') as f:
    json.dump(title_map, f, ensure_ascii=False, indent=2)

print(f"[INFO] Total items with title: {len(title_map)}")

[INFO] Total items with title: 182032


In [ ]:
import gzip
import json
from collections import defaultdict

# 输入输出路径
input_file = '/data/mhwang/LLM/Fre_LLM/dataset/raw_data/Movies.json.gz'
meta_file = '/data/mhwang/LLM/Fre_LLM/dataset/raw_data/meta_Movies.json'  # 修改：改为普通 .json
user_item_output = '/data/mhwang/LLM/Fre_LLM/dataset/process_data/Movies/user_item_5core_sorted.txt'
item_info_output = '/data/mhwang/LLM/Fre_LLM/dataset/process_data/Movies/item_info_5core.txt'

# Step 1: 统计每个用户和物品的交互次数
user_counts = defaultdict(int)
item_counts = defaultdict(int)

with gzip.open(input_file, 'rt', encoding='utf-8') as f:
    for line in f:
        data = json.loads(line.strip())
        user = data.get('reviewerID')
        item = data.get('asin')
        if user and item:
            user_counts[user] += 1
            item_counts[item] += 1

# Step 2: 筛选满足 >=5 core 的用户和物品
valid_users = set(u for u, c in user_counts.items() if c >= 5)
valid_items = set(i for i, c in item_counts.items() if c >= 5)

print(f"[INFO] Valid users: {len(valid_users)}")
print(f"[INFO] Valid items: {len(valid_items)}")

# Step 3: 加载元数据，获取每个 item 的 title（现在是普通 JSON 文件）
item_titles = {}  # item_id -> title

try:
    with open(meta_file, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                data = json.loads(line)
                item = data.get('asin')
                title = data.get('title', 'Unknown')
                if item:
                    # 防止 title 中出现换行符或空白字符
                    title = title.strip().replace('\n', ' ').replace('\r', '')
                    item_titles[item] = title if title else 'Unknown'
            except json.JSONDecodeError as e:
                print(f"[WARNING] JSON decode error in meta file: {e}")
                continue
except FileNotFoundError:
    print(f"[ERROR] Meta file not found: {meta_file}")
    raise

print(f"[INFO] Loaded titles for {len(item_titles)} items.")

# Step 4: 再次读取原始数据，收集有效交互记录（带时间戳）
user_interactions = defaultdict(list)

with gzip.open(input_file, 'rt', encoding='utf-8') as f:
    for line in f:
        data = json.loads(line.strip())
        user = data.get('reviewerID')
        item = data.get('asin')
        timestamp = data.get('unixReviewTime')

        if user in valid_users and item in valid_items and timestamp is not None:
            user_interactions[user].append((item, timestamp))

# Step 5: 对每个用户的交互按时间排序
sorted_user_interactions = {}
for user, interactions in user_interactions.items():
    sorted_interactions = sorted(interactions, key=lambda x: x[1])  # 按时间戳排序
    sorted_user_interactions[user] = [item for item, _ in sorted_interactions]

# Step 6: 映射 ID 并写入文件
user_map = {}
item_map = {}
user_counter = 1
item_counter = 1

with open(user_item_output, 'w', encoding='utf-8') as user_item_f, \
     open(item_info_output, 'w', encoding='utf-8') as item_info_f:

    for user, items in sorted_user_interactions.items():
        # 用户映射
        if user not in user_map:
            user_map[user] = user_counter
            user_counter += 1

        current_user_id = user_map[user]

        for item in items:
            # 物品映射
            if item not in item_map:
                item_map[item] = item_counter
                item_counter += 1

                # 获取 title（优先使用 meta 中的信息，否则 fallback 到 Unknown）
                title = item_titles.get(item, 'Unknown')
                item_info_f.write(f"{item_map[item]}::{title}\n")

            current_item_id = item_map[item]
            user_item_f.write(f"{current_user_id} {current_item_id}\n")

[INFO] Valid users: 98086
[INFO] Valid items: 62597
[INFO] Loaded titles for 181839 items.
